## Installation

In [8]:
!pip install langchain-groq -q
!pip install -qU langchain-google-genai
!pip install langchain langchain-community faiss-cpu tiktoken -q

In [9]:
import os

os.environ["GROQ_API_KEY"]   = "Your api key"
os.environ["GOOGLE_API_KEY"] = "Your api key"

print("Keys set.")

Keys set.


# Section 2: Core Components of LangChain


In [3]:
from langchain_groq import ChatGroq
import os

# LLM-style: invoke with a plain string
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

response = llm.invoke("Explain LangChain in simple terms")

print(response.content)

LangChain is a framework that helps developers build applications using large language models (LLMs) like chatbots, virtual assistants, or other AI-powered tools. 

Imagine you have a super smart, magic notebook that can understand and respond to anything you write in it. This notebook is like a large language model. However, to make the most of this notebook, you need a system to organize your thoughts, ask the right questions, and use the answers effectively. That's where LangChain comes in.

LangChain provides a set of tools and guidelines to help developers:

1. **Interact** with the magic notebook (large language model) in a structured way.
2. **Store** and **manage** the conversations, questions, and answers.
3. **Use** the answers to perform tasks, make decisions, or generate content.
4. **Improve** the overall performance of the application by fine-tuning the language model and the interaction process.

In simple terms, LangChain is like a blueprint or a set of instructions tha

In [4]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_groq import ChatGroq

chat_model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Chat Model-style: pass a list of typed messages
# SystemMessage → sets the persona/role
# HumanMessage  → the actual user question
messages = [
    SystemMessage(content="You are a helpful Senior Data Scientist."),
    HumanMessage(content="What is a Vector Store?")
]

response = chat_model.invoke(messages)

print(response.content)

A Vector Store, also known as a Vector Database or Vector Search Engine, is a type of database designed to efficiently store, index, and query large datasets of dense vectors, typically generated by machine learning models. These vectors are often used to represent complex data such as images, text, audio, or other types of multimedia content.

The primary goal of a Vector Store is to enable fast and accurate similarity searches, also known as nearest neighbor searches, within these high-dimensional vector spaces. This allows you to find the most similar items to a given query vector, which is useful in various applications, such as:

1. **Image and video search**: Find similar images or videos based on their visual features.
2. **Natural Language Processing (NLP)**: Search for similar text documents or sentences based on their semantic meaning.
3. **Recommendation systems**: Suggest items that are similar to a user's preferences or interests.
4. **Anomaly detection**: Identify unusual

## 2.2 Prompts and Prompt Templates


#### Basic PromptTemplate

In [5]:
from langchain_core.prompts import PromptTemplate

# Define a template with one variable
template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms"
)

# .format() fills placeholders → returns the final string
formatted_prompt = template.format(topic="neural networks")

print(formatted_prompt)

Explain neural networks in simple terms


#### ChatPromptTemplate

In [10]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "Explain {topic} simply")
])

# format_messages() returns a list of Message objects ready to send
messages = chat_template.format_messages(topic="overfitting")
print(messages)

[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain overfitting simply', additional_kwargs={}, response_metadata={})]


#### Few-Shot Prompt Templates

In [6]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Step 1: Define examples — teach the model the pattern
examples = [
    {"input": "I love this movie",  "output": "Positive"},
    {"input": "This is terrible",   "output": "Negative"},
    {"input": "Amazing experience", "output": "Positive"},
]

# Step 2: Template that formats each individual example
example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Text: {input}\nSentiment: {output}"
)

# Step 3: FewShot template: prefix + examples + suffix
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="Classify the sentiment of the following text:\n\n",
    suffix="Text: {input}\nSentiment:",
    input_variables=["input"]
)

# Step 4: Chain and invoke
chain = few_shot_prompt | model

response = chain.invoke({"input": "This product is amazing"})
print(response.content)

Positive.


## Type of Prompt Template

#### Basic PromptTemplate + LLM

In [11]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

prompt = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms"
)

chain = prompt | model

response = chain.invoke({"topic": "machine learning"})
print(response.content)

**Machine Learning in Simple Terms**

Machine learning is a way to teach computers to make decisions or predictions based on data, without being explicitly programmed. It's like teaching a child to recognize objects, but instead of using words, you show them many examples.

**How it Works:**

1. **Data Collection**: Gather lots of data, like images, text, or numbers.
2. **Pattern Recognition**: Use algorithms (like recipes) to find patterns in the data.
3. **Learning**: The computer uses these patterns to make predictions or decisions.
4. **Improvement**: The more data the computer sees, the better it becomes at making accurate predictions.

**Example:**

Imagine you want to teach a computer to recognize pictures of cats and dogs. You show it many pictures of cats and dogs, and it starts to notice patterns, like:

* Cats have pointy ears and whiskers.
* Dogs have floppy ears and tails.

The computer uses these patterns to make predictions: "This picture is a cat" or "This picture is a 

ChatPromptTemplate Example

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Step 1: Create chat prompt with system persona
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a teacher"),
    ("user", "Explain {topic} with examples")
])

# Step 2: Create chain
chain = chat_prompt | model

# Step 3: Invoke
response = chain.invoke({"topic": "overfitting"})
print(response.content)

Overfitting is a fundamental concept in machine learning and statistics that occurs when a model is too complex and learns the noise in the training data, rather than the underlying patterns. This results in the model performing well on the training data but poorly on new, unseen data.

**What is overfitting?**

Overfitting happens when a model is too closely fit to the training data, capturing the random fluctuations and noise in the data rather than the underlying relationships. As a result, the model becomes overly specialized to the training data and fails to generalize well to new data.

**Example 1: Polynomial Regression**

Suppose we want to model the relationship between the number of hours studied and the exam score. We collect data from 10 students and fit a polynomial regression model of degree 9 (a 9th-degree polynomial). The model fits the training data perfectly, but when we test it on new data, it performs poorly.

Training data:

| Hours Studied | Exam Score |
| --- | -

---
# Models

## LLM Example

In [13]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

response = llm.invoke("Explain machine learning in simple terms")

print(response.content)

**Machine Learning in Simple Terms**

Machine learning is a way to teach computers to make decisions or predictions based on data, without being explicitly programmed. It's like teaching a child to recognize objects, but instead of using words, you show them many examples.

**How it Works:**

1. **Data Collection**: Gather lots of data, like images, text, or numbers.
2. **Pattern Recognition**: Use algorithms (like recipes) to find patterns in the data.
3. **Learning**: The computer uses these patterns to make predictions or decisions.
4. **Improvement**: The more data the computer sees, the better it becomes at making predictions.

**Example:**

Imagine you want to teach a computer to recognize pictures of dogs and cats.

1. You show it many pictures of dogs and cats, labeled as "dog" or "cat".
2. The computer looks for patterns in the pictures, like shapes, colors, and textures.
3. The computer uses these patterns to predict whether a new picture is a dog or a cat.
4. As it sees more

## Chat Model Example

In [14]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_groq import ChatGroq

chat_model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

message = [
    SystemMessage(content="You are a helpful Sr Data Scientist."),
    HumanMessage(content="Explain overfitting")
]

response = chat_model.invoke(message)
print(response.content)

Overfitting is a fundamental concept in machine learning and data science. It occurs when a model is too complex and learns the noise in the training data, rather than the underlying patterns. As a result, the model performs well on the training data but poorly on new, unseen data.

**What happens during overfitting:**

1. **Model complexity**: The model has too many parameters or is too flexible, allowing it to fit the training data too closely.
2. **Noise fitting**: The model starts to fit the random fluctuations or noise in the training data, rather than the underlying relationships.
3. **Poor generalization**: The model becomes specialized to the training data and fails to generalize well to new data.

**Consequences of overfitting:**

1. **Poor predictive performance**: The model performs poorly on new, unseen data, leading to incorrect predictions or classifications.
2. **Lack of robustness**: The model is sensitive to small changes in the data or environment, making it unreliabl

### Embedding Example


In [15]:
!pip install -qU langchain-google-genai

In [16]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Initialize Gemini embedding model
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# embed_query converts text → list of floats (a vector)
vector = embeddings.embed_query("machine learning")

print(vector[:5])

[-0.006826791, 0.0032323739, 0.0036080698, -0.063535206, -0.016612697]


In [17]:
vec1 = embeddings.embed_query("cat")
print(vec1[:5])

[-0.017016973, 0.011458129, 0.022643944, -0.070259035, -0.0046942607]


In [18]:
vec2 = embeddings.embed_query("dog")
print(vec2[:5])

[-0.0152275665, -0.013213994, 0.028866338, -0.06026992, -0.014451638]


In [19]:
from sklearn.metrics.pairwise import cosine_similarity

# cosine_similarity → value between 0 and 1
# 1.0 = identical meaning, 0.0 = completely unrelated
similarity = cosine_similarity([vec1], [vec2])
print(similarity)
# Expected: ~0.74 (cat and dog are semantically close)

[[0.74689359]]


## 2.3 Output Parsers


In [20]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_groq import ChatGroq

parser = JsonOutputParser()

model = ChatGroq(model="llama-3.3-70b-versatile")

response = model.invoke("Return name and age in JSON format")

# JsonOutputParser converts AIMessage → Python dict
parsed_output = parser.invoke(response)
print(parsed_output)

{'name': 'John Doe', 'age': 30}



# Section 4: Hands-on Code Examples

## 4.1 Basic LLM Call

In [21]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# Step 1: Create model
# temperature=0 → deterministic, factual output
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Step 2: Create prompt template
prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple terms"
)

# Step 3: Create chain using LCEL pipe operator
chain = prompt | model

# Step 4: Run chain — response is an AIMessage object
response = chain.invoke({"topic": "neural networks"})

print(response.content)

**What is a Neural Network.** 
A neural network is a computer system that works like the human brain. It's made up of many interconnected nodes (called "neurons") that process and transmit information.

**How it Works:**
1. **Input**: The network receives data, like images or sounds.
2. **Processing**: The data is sent through multiple layers of neurons, which analyze and transform it.
3. **Output**: The final layer produces a result, like a prediction or classification.

**Key Concepts:**

* **Artificial Neurons**: These are the basic building blocks of the network. They receive input, perform a calculation, and send the result to other neurons.
* **Layers**: Neural networks are organized into layers, each with its own role (e.g., input, hidden, output).
* **Connections**: Neurons are connected, allowing them to exchange information.
* **Training**: The network learns from data by adjusting the connections between neurons.

**Simple Analogy:**
Imagine a team of experts working togethe

---
## 4.2 Simple Chain

Simple Chain Example

In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

prompt = ChatPromptTemplate.from_template("Explain {topic} simply")

model = ChatGroq(model="llama-3.3-70b-versatile")

chain = prompt | model

response = chain.invoke({"topic": "AI"})
print(response.content)

**What is AI.** 
AI stands for Artificial Intelligence. It's a type of computer science that enables machines to think and learn like humans.

**How does it work.** 
Imagine you're trying to teach a child to recognize different animals. You show them pictures of cats and dogs, and they start to understand the differences. AI works in a similar way. It uses data (like pictures, text, or sounds) to learn and make decisions.

**Key concepts:**

1. **Machine Learning**: AI learns from data, like a child learning from examples.
2. **Neural Networks**: AI uses complex networks to process information, like the human brain.
3. **Algorithms**: AI follows rules and instructions to make decisions.

**Examples of AI:**

1. **Virtual Assistants**: Siri, Alexa, or Google Assistant, which understand voice commands.
2. **Image Recognition**: Facebook's ability to tag friends in photos.
3. **Self-Driving Cars**: Cars that can navigate roads and make decisions without human input.

**In simple terms**, 

#### LCEL Example — Full Pipeline with Output Parser (Most Important)

In [23]:
from langchain_core.output_parsers import StrOutputParser

# Full LCEL pipeline: prompt → model → parser
# StrOutputParser extracts .content from AIMessage → returns plain string
chain = prompt | model | StrOutputParser()

result = chain.invoke({"topic": "Machine Learning"})
print(result)

**Machine Learning in Simple Terms**

Machine Learning is a way to teach computers to learn from data and make predictions or decisions without being explicitly programmed. It's like teaching a child to recognize animals:

1. **Show examples**: You show the child many pictures of different animals, such as cats, dogs, and birds.
2. **Learn patterns**: The child starts to recognize patterns and features that distinguish each animal, like the shape of the ears or the color of the fur.
3. **Make predictions**: When the child sees a new picture of an animal, they can use what they've learned to predict what kind of animal it is.

**How Machine Learning Works**

1. **Data collection**: Gather a large amount of data, such as images, text, or numbers.
2. **Training**: Use the data to train a computer model to recognize patterns and relationships.
3. **Testing**: Test the model on new, unseen data to see how well it performs.
4. **Refining**: Refine the model by adjusting its parameters and re


## 4.3 Memory Example


In [34]:
!pip install langchain-groq -q
!pip install -qU langchain-google-genai
!pip install langchain langchain-community faiss-cpu tiktoken -q

In [37]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Modern memory — plain list holds the full conversation history
chat_history = []

# MessagesPlaceholder injects the full history into every prompt automatically
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

chain = prompt | model

def chat(user_input):
    # Store user message
    chat_history.append(HumanMessage(content=user_input))
    # Run chain with full history injected
    response = chain.invoke({
        "input": user_input,
        "chat_history": chat_history
    })
    # Store AI reply
    chat_history.append(AIMessage(content=response.content))
    return response.content

# Turn 1: introduce name
r1 = chat("Hi! My name is Prasad. I am a Data Science intern at Innomatics.")
print("Turn 1:", r1)

# Turn 2: ask something unrelated
r2 = chat("What is gradient descent?")
print("\nTurn 2:", r2)

# Turn 3: memory test — model should remember the name
r3 = chat("What is my name and where do I intern?")
print("\nTurn 3 (memory test):", r3)

# See what is stored in memory
print("\n--- Memory Buffer ---")
for msg in chat_history:
    role = "Human" if isinstance(msg, HumanMessage) else "AI"
    print(f"[{role}]: {msg.content[:120]}")

Turn 1: Hello Prasad, nice to meet you. Congratulations on your Data Science internship at Innomatics. How's your experience been so far? What kind of projects are you working on?

Turn 2: **Gradient Descent**

Gradient descent is a fundamental optimization algorithm in machine learning and deep learning. It's used to minimize the loss function of a model by iteratively adjusting the model's parameters in the direction of the negative gradient of the loss function.

**How Gradient Descent Works**
-----------------------------

1. **Initialize parameters**: The model's parameters are initialized with random values.
2. **Compute loss**: The loss function is computed using the current parameters and the training data.
3. **Compute gradient**: The gradient of the loss function with respect to each parameter is computed.
4. **Update parameters**: The parameters are updated by subtracting the product of the learning rate and the gradient from the current parameters.
5. **Repeat**: Steps 2-4 

## 4.4 Agent with Tool


In [41]:
!pip install langchain -q

In [43]:
!pip install langgraph -q

In [47]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# ------------------------------------------------------------------ #
# Define tools using the @tool decorator                              #
# The docstring is what the agent reads to decide when to use it      #
# ------------------------------------------------------------------ #

@tool
def calculate_bmi(weight_kg: float, height_m: float) -> str:
    """Calculate Body Mass Index (BMI) given weight in kg and height in meters."""
    bmi = weight_kg / (height_m ** 2)
    if bmi < 18.5:
        category = "Underweight"
    elif bmi < 25:
        category = "Normal weight"
    elif bmi < 30:
        category = "Overweight"
    else:
        category = "Obese"
    return f"BMI: {bmi:.2f} — Category: {category}"


@tool
def celsius_to_fahrenheit(celsius: float) -> str:
    """Convert a temperature value from Celsius to Fahrenheit."""
    fahrenheit = (celsius * 9 / 5) + 32
    return f"{celsius}°C = {fahrenheit:.1f}°F"


tools = [calculate_bmi, celsius_to_fahrenheit]

# langgraph's create_react_agent handles the ReAct loop internally
# no manual PromptTemplate needed — just pass model + tools
agent = create_react_agent(model=model, tools=tools)

# Run the agent
result = agent.invoke({
    "messages": [
        ("human", "What is the BMI of someone who weighs 70 kg and is 1.75 m tall? Also convert 37 Celsius to Fahrenheit.")
    ]
})

print("\n=== Final Answer ===")
print(result["messages"][-1].content)

/tmp/ipykernel_1449/2526101553.py:38: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model=model, tools=tools)



=== Final Answer ===
The BMI of someone who weighs 70 kg and is 1.75 m tall is 22.86, which falls under the category of normal weight. The temperature 37 Celsius is equivalent to 98.6 Fahrenheit.


## 4.5 Document Loaders & Vector Stores (RAG Pipeline)


In [52]:
!pip install langchain-text-splitters -q

In [56]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter  # ← only this line changed

raw_documents = [
    Document(page_content="""
        LangChain is an open-source framework for building LLM-powered applications.
        It was created by Harrison Chase and first released in October 2022.
        LangChain provides tools for chaining prompts, managing memory, building
        agents, and connecting LLMs to external data sources and vector stores.
    """, metadata={"source": "langchain_intro.txt"}),

    Document(page_content="""
        FAISS (Facebook AI Similarity Search) is a library for fast similarity search
        over dense vectors. In LangChain it is used as a local vector store that runs
        entirely in memory, making it ideal for prototyping. FAISS indexes vectors
        and retrieves the nearest neighbours in milliseconds.
    """, metadata={"source": "faiss_info.txt"}),

    Document(page_content="""
        RAG stands for Retrieval-Augmented Generation. In a RAG pipeline, the user
        query is embedded and used to search a vector store for the most relevant
        document chunks. Those chunks are injected into the LLM prompt as context,
        allowing the model to answer questions grounded in real retrieved text,
        which significantly reduces hallucinations.
    """, metadata={"source": "rag_explained.txt"}),
]

print(f"Loaded {len(raw_documents)} documents.")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(raw_documents)
print(f"Split into {len(chunks)} chunks.")

Loaded 3 documents.
Split into 6 chunks.


In [57]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vector_store = FAISS.from_documents(chunks, embeddings)

print("Vector store built successfully.")
print(f"Total vectors indexed: {vector_store.index.ntotal}")

Vector store built successfully.
Total vectors indexed: 6


In [59]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# ------------------------------------------------------------------ #
# Build retriever from the vector store created in the previous cell  #
# k=2 → fetch top 2 most relevant chunks per query                    #
# ------------------------------------------------------------------ #
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Prompt that injects retrieved chunks as context
prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Modern LCEL RAG chain — replaces RetrievalQA
qa_chain = (
    {
        "context":  retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

# Ask questions — answered from our documents, not LLM training data
questions = [
    "What is RAG and how does it reduce hallucinations?",
    "Who created LangChain and when?",
    "What is FAISS used for in LangChain?"
]

for q in questions:
    answer = qa_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 60)

Q: What is RAG and how does it reduce hallucinations?
A: RAG stands for Retrieval-Augmented Generation. It reduces hallucinations by allowing the model to answer questions grounded in real retrieved text. In a RAG pipeline, the user query is used to search for the most relevant document chunks, which are then injected into the prompt as context.
------------------------------------------------------------
Q: Who created LangChain and when?
A: LangChain was created by Harrison Chase and was first released in October 2022.
------------------------------------------------------------
Q: What is FAISS used for in LangChain?
A: FAISS is used as a local vector store that runs entirely in memory in LangChain, making it ideal for prototyping.
------------------------------------------------------------



# Section 5: Real-World Use Cases

## Use Case 1: Customer Support Chatbot with Memory


In [61]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.5)

# Modern memory — plain list holds full conversation history
chat_history = []

support_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly customer support agent for TechStore India. "
               "Help with orders, returns, and product queries. Be polite and concise."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

chain = support_prompt | model

def support_chat(user_input):
    chat_history.append(HumanMessage(content=user_input))
    response = chain.invoke({
        "input": user_input,
        "chat_history": chat_history
    })
    chat_history.append(AIMessage(content=response.content))
    return response.content

print("=" * 60)
print("TechStore India — Customer Support Chat")
print("=" * 60)

turns = [
    "Hi, my name is Prasad. I placed an order 3 days ago but haven't received it.",
    "My order ID is TS-2026-4892.",
    "Also, what is your return policy?",
    "What was my order ID again?",    # memory test
]

for turn in turns:
    response = support_chat(turn)
    print(f"\nCustomer: {turn}")
    print(f"Support Agent: {response}")

print("\n" + "=" * 60)

TechStore India — Customer Support Chat

Customer: Hi, my name is Prasad. I placed an order 3 days ago but haven't received it.
Support Agent: Hello Prasad, thank you for reaching out to TechStore India. I'd be happy to help you with your order. Can you please provide me with your order number so I can look into the status of your shipment?

Customer: My order ID is TS-2026-4892.
Support Agent: Thank you, Prasad. I've checked on the status of your order TS-2026-4892. It seems that your order was shipped out on the same day it was placed, and the estimated delivery time is 3-5 business days. Since it's only been 3 days, it's possible that the order is still in transit. However, I can try to track the shipment for you. Would you like me to do that and provide you with an update?

Customer: Also, what is your return policy?
Support Agent: At TechStore India, we have a 7-day return policy. If you're not satisfied with your purchase, you can return it within 7 days of delivery. The item mus